# Testing Accessibility Knowledge Across Pythia Model Sizes

## Setup

In [33]:
%pip install transformer_lens -q
%pip install circuitsvis -q

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [1]:
from transformer_lens import HookedTransformer
import transformer_lens.utils as utils
import csv
import torch
import gc
import circuitsvis as cv

## Pythia 160M

### Load Model

In [2]:
model = HookedTransformer.from_pretrained("pythia-160m")
print(f"Layers: {model.cfg.n_layers}")
print(f"Heads: {model.cfg.n_heads}")
print(f"Hidden size: {model.cfg.d_model}")
print(f"Params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model pythia-160m into HookedTransformer
Layers: 12
Heads: 12
Hidden size: 768
Params: 162.3M


### Attention Pattern Analysis

#### Screen Reader (reader to screen)

In [3]:
# DIRECTION: second token attends BACK to first token
# "reader" looks at "screen" = attn[reader_idx, screen_idx]
# "text" looks at "alt" = attn[text_idx, alt_idx]
# "link" looks at "skip" = attn[link_idx, skip_idx]
# Rule: attn[SECOND, FIRST]

prompt = "A screen reader is"
tokens = model.to_str_tokens(prompt)
print(list(enumerate(tokens)))  # verify indices
logits, cache = model.run_with_cache(prompt)

threshold = 0.1
rows = []

for layer in range(model.cfg.n_layers):
    attention = cache["pattern", layer]
    for head in range(model.cfg.n_heads):
        attn = attention[0, head]
        reader_idx = 3
        screen_idx = 2
        score = attn[reader_idx, screen_idx].item()
        if score > threshold:
            rows.append({
                "layer": layer,
                "head": head,
                "binding_score": round(score, 4)
            })

with open("../../results/pythia/extended/160m_screenReader_attention_binding.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["layer", "head", "binding_score"])
    writer.writeheader()
    writer.writerows(rows)

print(f"Found {len(rows)} heads above threshold {threshold}")
print("\nTop 10 by binding strength:")
sorted_rows = sorted(rows, key=lambda x: x["binding_score"], reverse=True)
for row in sorted_rows[:10]:
    print(f"Layer {row['layer']:2d}, Head {row['head']:2d}: {row['binding_score']}")
print("Saved to 160m_screenReader_attention_binding.csv")

[(0, '<|endoftext|>'), (1, 'A'), (2, ' screen'), (3, ' reader'), (4, ' is')]
Found 46 heads above threshold 0.1

Top 10 by binding strength:
Layer 11, Head  8: 1.0
Layer 11, Head  6: 0.9712
Layer  3, Head  0: 0.9243
Layer  3, Head  2: 0.9156
Layer  1, Head  1: 0.7337
Layer  7, Head  7: 0.6743
Layer  0, Head  0: 0.6281
Layer  1, Head  2: 0.6271
Layer  2, Head 11: 0.6195
Layer  2, Head  8: 0.5459
Saved to 160m_screenReader_attention_binding.csv


##### Circuitsvis
*Circuitsvis* is run on the layer that has the head with the top binding strength.
**Layer 11, Head 8 (score: 1.0)**

In [4]:
# Layer 11, Head 8 (score: 1.0)
layer = 11
attention = cache["pattern", layer]
cv.attention.attention_patterns(tokens=tokens, attention=attention[0])

#### Alt Text (text to alt)

In [5]:
# DIRECTION: "text" looks at "alt" = attn[text_idx, alt_idx]
# Rule: attn[SECOND, FIRST]

prompt = "An image needs alt text to be accessible"
tokens = model.to_str_tokens(prompt)
print(list(enumerate(tokens)))  # verify indices
logits, cache = model.run_with_cache(prompt)

threshold = 0.1
rows = []

for layer in range(model.cfg.n_layers):
    attention = cache["pattern", layer]
    for head in range(model.cfg.n_heads):
        attn = attention[0, head]
        text_idx = 5
        alt_idx = 4
        score = attn[text_idx, alt_idx].item()
        if score > threshold:
            rows.append({
                "layer": layer,
                "head": head,
                "binding_score": round(score, 4)
            })

with open("../../results/pythia/extended/160m_altText_attention_binding.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["layer", "head", "binding_score"])
    writer.writeheader()
    writer.writerows(rows)

print(f"Found {len(rows)} heads above threshold {threshold}")
print("\nTop 10 by binding strength:")
sorted_rows = sorted(rows, key=lambda x: x["binding_score"], reverse=True)
for row in sorted_rows[:10]:
    print(f"Layer {row['layer']:2d}, Head {row['head']:2d}: {row['binding_score']}")
print("Saved to 160m_altText_attention_binding.csv")

[(0, '<|endoftext|>'), (1, 'An'), (2, ' image'), (3, ' needs'), (4, ' alt'), (5, ' text'), (6, ' to'), (7, ' be'), (8, ' accessible')]
Found 39 heads above threshold 0.1

Top 10 by binding strength:
Layer 11, Head  0: 0.9932
Layer  3, Head  0: 0.9509
Layer  0, Head  2: 0.9273
Layer  3, Head  2: 0.9091
Layer  1, Head  1: 0.8106
Layer  2, Head  5: 0.6864
Layer  7, Head  7: 0.6797
Layer  6, Head  2: 0.6661
Layer  1, Head  2: 0.6371
Layer  0, Head  8: 0.6308
Saved to 160m_altText_attention_binding.csv


##### Circuitsvis
**Layer 11, Head 0 (score: 0.9932)**

In [6]:
# Layer 11, Head 0 (score: 0.9932)
layer = 11
attention = cache["pattern", layer]
cv.attention.attention_patterns(tokens=tokens, attention=attention[0])

#### Skip Link (link to skip)

In [8]:
# DIRECTION: "link" looks at "skip" = attn[link_idx, skip_idx]
# Rule: attn[SECOND, FIRST]

prompt = "Use a skip link to bypass navigation"
tokens = model.to_str_tokens(prompt)
print(list(enumerate(tokens)))  # verify indices
logits, cache = model.run_with_cache(prompt)

threshold = 0.1
rows = []

for layer in range(model.cfg.n_layers):
    attention = cache["pattern", layer]
    for head in range(model.cfg.n_heads):
        attn = attention[0, head]
        link_idx = 4
        skip_idx = 3
        score = attn[link_idx, skip_idx].item()
        if score > threshold:
            rows.append({
                "layer": layer,
                "head": head,
                "binding_score": round(score, 4)
            })

with open("../../results/pythia/extended/160m_skipLink_attention_binding.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["layer", "head", "binding_score"])
    writer.writeheader()
    writer.writerows(rows)

print(f"Found {len(rows)} heads above threshold {threshold}")
print("\nTop 10 by binding strength:")
sorted_rows = sorted(rows, key=lambda x: x["binding_score"], reverse=True)
for row in sorted_rows[:10]:
    print(f"Layer {row['layer']:2d}, Head {row['head']:2d}: {row['binding_score']}")
print("Saved to 160m_skipLink_attention_binding.csv")

[(0, '<|endoftext|>'), (1, 'Use'), (2, ' a'), (3, ' skip'), (4, ' link'), (5, ' to'), (6, ' bypass'), (7, ' navigation')]
Found 47 heads above threshold 0.1

Top 10 by binding strength:
Layer 11, Head  0: 1.0
Layer 11, Head  6: 1.0
Layer  3, Head  2: 0.9774
Layer  3, Head  0: 0.9526
Layer 10, Head  1: 0.9244
Layer  6, Head  2: 0.9175
Layer  7, Head  7: 0.8944
Layer 10, Head  9: 0.8054
Layer  2, Head  8: 0.702
Layer  2, Head  9: 0.7013
Saved to 160m_skipLink_attention_binding.csv


##### Circuitsvis
**Layer 11, Head 0 & Head 6 (score: 1.0)**

In [9]:
# Layer 11, Head 0 & Head 6 (score: 1.0)
layer = 11
attention = cache["pattern", layer]
cv.attention.attention_patterns(tokens=tokens, attention=attention[0])

### Delete Model & Clear Cache

In [10]:
# Run this between models
del model
del cache
gc.collect()
torch.cuda.empty_cache()
print("Memory cleared")

Memory cleared


## Pythia 410M

### Load Model

In [11]:
model = HookedTransformer.from_pretrained("pythia-410m")
print(f"Layers: {model.cfg.n_layers}")
print(f"Heads: {model.cfg.n_heads}")
print(f"Hidden size: {model.cfg.d_model}")
print(f"Params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

Loaded pretrained model pythia-410m into HookedTransformer
Layers: 24
Heads: 16
Hidden size: 1024
Params: 405.3M


### Attention Pattern Analysis

#### Screen Reader (reader to screen)

In [12]:
# DIRECTION: second token attends BACK to first token
# "reader" looks at "screen" = attn[reader_idx, screen_idx]
# "text" looks at "alt" = attn[text_idx, alt_idx]
# "link" looks at "skip" = attn[link_idx, skip_idx]
# Rule: attn[SECOND, FIRST]

prompt = "A screen reader is"
tokens = model.to_str_tokens(prompt)
print(list(enumerate(tokens)))  # verify indices
logits, cache = model.run_with_cache(prompt)

threshold = 0.1
rows = []

for layer in range(model.cfg.n_layers):
    attention = cache["pattern", layer]
    for head in range(model.cfg.n_heads):
        attn = attention[0, head]
        reader_idx = 3
        screen_idx = 2
        score = attn[reader_idx, screen_idx].item()
        if score > threshold:
            rows.append({
                "layer": layer,
                "head": head,
                "binding_score": round(score, 4)
            })

with open("../../results/pythia/extended/410m_screenReader_attention_binding.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["layer", "head", "binding_score"])
    writer.writeheader()
    writer.writerows(rows)

print(f"Found {len(rows)} heads above threshold {threshold}")
print("\nTop 10 by binding strength:")
sorted_rows = sorted(rows, key=lambda x: x["binding_score"], reverse=True)
for row in sorted_rows[:10]:
    print(f"Layer {row['layer']:2d}, Head {row['head']:2d}: {row['binding_score']}")
print("Saved to 410m_screenReader_attention_binding.csv")

[(0, '<|endoftext|>'), (1, 'A'), (2, ' screen'), (3, ' reader'), (4, ' is')]
Found 71 heads above threshold 0.1

Top 10 by binding strength:
Layer  0, Head 14: 0.9758
Layer  3, Head 13: 0.9591
Layer  1, Head  3: 0.9404
Layer  3, Head  3: 0.8673
Layer  5, Head  2: 0.8562
Layer  3, Head  4: 0.8519
Layer  0, Head  0: 0.8454
Layer  1, Head 10: 0.8005
Layer  3, Head  5: 0.7801
Layer  2, Head 13: 0.7638
Saved to 410m_screenReader_attention_binding.csv


##### Circuitsvis
**Layer 0, Head 14**

In [13]:
# Layer 0, Head 14
layer = 0
attention = cache["pattern", layer]
cv.attention.attention_patterns(tokens=tokens, attention=attention[0])

#### Alt Text (text to alt)

In [15]:
# DIRECTION: "text" looks at "alt" = attn[text_idx, alt_idx]
# Rule: attn[SECOND, FIRST]

prompt = "An image needs alt text to be accessible"
tokens = model.to_str_tokens(prompt)
print(list(enumerate(tokens)))  # verify indices
logits, cache = model.run_with_cache(prompt)

threshold = 0.1
rows = []

for layer in range(model.cfg.n_layers):
    attention = cache["pattern", layer]
    for head in range(model.cfg.n_heads):
        attn = attention[0, head]
        text_idx = 5
        alt_idx = 4
        score = attn[text_idx, alt_idx].item()
        if score > threshold:
            rows.append({
                "layer": layer,
                "head": head,
                "binding_score": round(score, 4)
            })

with open("../../results/pythia/extended/410m_altText_attention_binding.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["layer", "head", "binding_score"])
    writer.writeheader()
    writer.writerows(rows)

print(f"Found {len(rows)} heads above threshold {threshold}")
print("\nTop 10 by binding strength:")
sorted_rows = sorted(rows, key=lambda x: x["binding_score"], reverse=True)
for row in sorted_rows[:10]:
    print(f"Layer {row['layer']:2d}, Head {row['head']:2d}: {row['binding_score']}")
print("Saved to 410m_altText_attention_binding.csv")

[(0, '<|endoftext|>'), (1, 'An'), (2, ' image'), (3, ' needs'), (4, ' alt'), (5, ' text'), (6, ' to'), (7, ' be'), (8, ' accessible')]
Found 96 heads above threshold 0.1

Top 10 by binding strength:
Layer  4, Head  9: 0.9677
Layer  3, Head  1: 0.9601
Layer  5, Head  2: 0.9386
Layer  3, Head  3: 0.7565
Layer  9, Head  5: 0.7555
Layer  0, Head 14: 0.7371
Layer  5, Head  3: 0.7303
Layer  1, Head  6: 0.7279
Layer  5, Head  5: 0.7047
Layer  1, Head 10: 0.6733
Saved to 410m_altText_attention_binding.csv


##### Circuitsvis
Layer 4 Head 9

In [16]:
# TODO: update layer after running analysis above
layer = 4
attention = cache["pattern", layer]
cv.attention.attention_patterns(tokens=tokens, attention=attention[0])

#### Skip Link (link to skip)

In [17]:
# DIRECTION: "link" looks at "skip" = attn[link_idx, skip_idx]
# Rule: attn[SECOND, FIRST]

prompt = "Use a skip link to bypass navigation"
tokens = model.to_str_tokens(prompt)
print(list(enumerate(tokens)))  # verify indices
logits, cache = model.run_with_cache(prompt)

threshold = 0.1
rows = []

for layer in range(model.cfg.n_layers):
    attention = cache["pattern", layer]
    for head in range(model.cfg.n_heads):
        attn = attention[0, head]
        link_idx = 4
        skip_idx = 3
        score = attn[link_idx, skip_idx].item()
        if score > threshold:
            rows.append({
                "layer": layer,
                "head": head,
                "binding_score": round(score, 4)
            })

with open("../../results/pythia/extended/410m_skipLink_attention_binding.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["layer", "head", "binding_score"])
    writer.writeheader()
    writer.writerows(rows)

print(f"Found {len(rows)} heads above threshold {threshold}")
print("\nTop 10 by binding strength:")
sorted_rows = sorted(rows, key=lambda x: x["binding_score"], reverse=True)
for row in sorted_rows[:10]:
    print(f"Layer {row['layer']:2d}, Head {row['head']:2d}: {row['binding_score']}")
print("Saved to 410m_skipLink_attention_binding.csv")

[(0, '<|endoftext|>'), (1, 'Use'), (2, ' a'), (3, ' skip'), (4, ' link'), (5, ' to'), (6, ' bypass'), (7, ' navigation')]
Found 81 heads above threshold 0.1

Top 10 by binding strength:
Layer 22, Head  8: 0.9993
Layer 22, Head  2: 0.999
Layer 23, Head  6: 0.999
Layer 21, Head  2: 0.9205
Layer  3, Head  1: 0.9128
Layer  5, Head  2: 0.8853
Layer  4, Head  9: 0.8613
Layer 22, Head  6: 0.835
Layer 22, Head 10: 0.8177
Layer 23, Head  8: 0.791
Saved to 410m_skipLink_attention_binding.csv


##### Circuitsvis
Layer 22 Head 8

In [18]:
# TODO: update layer after running analysis above
layer = 22
attention = cache["pattern", layer]
cv.attention.attention_patterns(tokens=tokens, attention=attention[0])

### Delete Model & Clear Cache

In [19]:
# Run this between models
del model
del cache
gc.collect()
torch.cuda.empty_cache()
print("Memory cleared")

Memory cleared


## Pythia 1B

### Load Model

In [20]:
model = HookedTransformer.from_pretrained("pythia-1b")
print(f"Layers: {model.cfg.n_layers}")
print(f"Heads: {model.cfg.n_heads}")
print(f"Hidden size: {model.cfg.d_model}")
print(f"Params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

Loaded pretrained model pythia-1b into HookedTransformer
Layers: 16
Heads: 8
Hidden size: 2048
Params: 1011.7M


### Attention Pattern Analysis

#### Screen Reader (reader to screen)

In [21]:
# DIRECTION: second token attends BACK to first token
# "reader" looks at "screen" = attn[reader_idx, screen_idx]
# "text" looks at "alt" = attn[text_idx, alt_idx]
# "link" looks at "skip" = attn[link_idx, skip_idx]
# Rule: attn[SECOND, FIRST]

prompt = "A screen reader is"
tokens = model.to_str_tokens(prompt)
print(list(enumerate(tokens)))  # verify indices
logits, cache = model.run_with_cache(prompt)

threshold = 0.1
rows = []

for layer in range(model.cfg.n_layers):
    attention = cache["pattern", layer]
    for head in range(model.cfg.n_heads):
        attn = attention[0, head]
        reader_idx = 3
        screen_idx = 2
        score = attn[reader_idx, screen_idx].item()
        if score > threshold:
            rows.append({
                "layer": layer,
                "head": head,
                "binding_score": round(score, 4)
            })

with open("../../results/pythia/extended/1b_screenReader_attention_binding.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["layer", "head", "binding_score"])
    writer.writeheader()
    writer.writerows(rows)

print(f"Found {len(rows)} heads above threshold {threshold}")
print("\nTop 10 by binding strength:")
sorted_rows = sorted(rows, key=lambda x: x["binding_score"], reverse=True)
for row in sorted_rows[:10]:
    print(f"Layer {row['layer']:2d}, Head {row['head']:2d}: {row['binding_score']}")
print("Saved to 1b_screenReader_attention_binding.csv")

[(0, '<|endoftext|>'), (1, 'A'), (2, ' screen'), (3, ' reader'), (4, ' is')]
Found 28 heads above threshold 0.1

Top 10 by binding strength:
Layer  3, Head  5: 0.9932
Layer  1, Head  4: 0.9704
Layer  0, Head  3: 0.9656
Layer  1, Head  0: 0.8892
Layer  3, Head  6: 0.7256
Layer  2, Head  2: 0.7
Layer  1, Head  3: 0.6673
Layer  2, Head  0: 0.661
Layer  6, Head  3: 0.645
Layer  1, Head  2: 0.5844
Saved to 1b_screenReader_attention_binding.csv


##### Circuitsvis
**Layer 3, Head 5**

In [22]:
# Layer 3, Head 5
layer = 3
attention = cache["pattern", layer]
cv.attention.attention_patterns(tokens=tokens, attention=attention[0])

#### Alt Text (text to alt)

In [23]:
# DIRECTION: "text" looks at "alt" = attn[text_idx, alt_idx]
# Rule: attn[SECOND, FIRST]

prompt = "An image needs alt text to be accessible"
tokens = model.to_str_tokens(prompt)
print(list(enumerate(tokens)))  # verify indices
logits, cache = model.run_with_cache(prompt)

threshold = 0.1
rows = []

for layer in range(model.cfg.n_layers):
    attention = cache["pattern", layer]
    for head in range(model.cfg.n_heads):
        attn = attention[0, head]
        text_idx = 5
        alt_idx = 4
        score = attn[text_idx, alt_idx].item()
        if score > threshold:
            rows.append({
                "layer": layer,
                "head": head,
                "binding_score": round(score, 4)
            })

with open("../../results/pythia/extended/1b_altText_attention_binding.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["layer", "head", "binding_score"])
    writer.writeheader()
    writer.writerows(rows)

print(f"Found {len(rows)} heads above threshold {threshold}")
print("\nTop 10 by binding strength:")
sorted_rows = sorted(rows, key=lambda x: x["binding_score"], reverse=True)
for row in sorted_rows[:10]:
    print(f"Layer {row['layer']:2d}, Head {row['head']:2d}: {row['binding_score']}")
print("Saved to 1b_altText_attention_binding.csv")

[(0, '<|endoftext|>'), (1, 'An'), (2, ' image'), (3, ' needs'), (4, ' alt'), (5, ' text'), (6, ' to'), (7, ' be'), (8, ' accessible')]
Found 47 heads above threshold 0.1

Top 10 by binding strength:
Layer  3, Head  5: 0.9997
Layer  1, Head  4: 0.9796
Layer  3, Head  6: 0.8934
Layer  6, Head  3: 0.8619
Layer  0, Head  3: 0.6822
Layer  1, Head  3: 0.6288
Layer  3, Head  7: 0.619
Layer  2, Head  2: 0.5325
Layer  2, Head  0: 0.4798
Layer  1, Head  0: 0.4701
Saved to 1b_altText_attention_binding.csv


##### Circuitsvis
Layer 3 Head 5

In [25]:
# TODO: update layer after running analysis above
layer = 3
attention = cache["pattern", layer]
cv.attention.attention_patterns(tokens=tokens, attention=attention[0])

#### Skip Link (link to skip)

In [26]:
# DIRECTION: "link" looks at "skip" = attn[link_idx, skip_idx]
# Rule: attn[SECOND, FIRST]

prompt = "Use a skip link to bypass navigation"
tokens = model.to_str_tokens(prompt)
print(list(enumerate(tokens)))  # verify indices
logits, cache = model.run_with_cache(prompt)

threshold = 0.1
rows = []

for layer in range(model.cfg.n_layers):
    attention = cache["pattern", layer]
    for head in range(model.cfg.n_heads):
        attn = attention[0, head]
        link_idx = 4
        skip_idx = 3
        score = attn[link_idx, skip_idx].item()
        if score > threshold:
            rows.append({
                "layer": layer,
                "head": head,
                "binding_score": round(score, 4)
            })

with open("../../results/pythia/extended/1b_skipLink_attention_binding.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["layer", "head", "binding_score"])
    writer.writeheader()
    writer.writerows(rows)

print(f"Found {len(rows)} heads above threshold {threshold}")
print("\nTop 10 by binding strength:")
sorted_rows = sorted(rows, key=lambda x: x["binding_score"], reverse=True)
for row in sorted_rows[:10]:
    print(f"Layer {row['layer']:2d}, Head {row['head']:2d}: {row['binding_score']}")
print("Saved to 1b_skipLink_attention_binding.csv")

[(0, '<|endoftext|>'), (1, 'Use'), (2, ' a'), (3, ' skip'), (4, ' link'), (5, ' to'), (6, ' bypass'), (7, ' navigation')]
Found 35 heads above threshold 0.1

Top 10 by binding strength:
Layer  3, Head  5: 0.9994
Layer  3, Head  6: 0.9288
Layer  6, Head  3: 0.8928
Layer  1, Head  4: 0.8488
Layer  0, Head  3: 0.7983
Layer  1, Head  3: 0.7668
Layer  3, Head  7: 0.6938
Layer  2, Head  0: 0.4117
Layer  2, Head  6: 0.4078
Layer  2, Head  4: 0.3557
Saved to 1b_skipLink_attention_binding.csv


##### Circuitsvis
Layer 3 Head 5

In [28]:
# TODO: update layer after running analysis above
layer = 3
attention = cache["pattern", layer]
cv.attention.attention_patterns(tokens=tokens, attention=attention[0])

### Delete Model & Clear Cache

In [29]:
# Run this between models
del model
del cache
gc.collect()
torch.cuda.empty_cache()
print("Memory cleared")

Memory cleared


## Pythia 2.8B

### Load Model

In [30]:
model = HookedTransformer.from_pretrained("pythia-2.8b")
print(f"Layers: {model.cfg.n_layers}")
print(f"Heads: {model.cfg.n_heads}")
print(f"Hidden size: {model.cfg.d_model}")
print(f"Params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

Loaded pretrained model pythia-2.8b into HookedTransformer
Layers: 32
Heads: 32
Hidden size: 2560
Params: 2774.9M


### Attention Pattern Analysis

#### Screen Reader (reader to screen)

In [31]:
# DIRECTION: second token attends BACK to first token
# "reader" looks at "screen" = attn[reader_idx, screen_idx]
# "text" looks at "alt" = attn[text_idx, alt_idx]
# "link" looks at "skip" = attn[link_idx, skip_idx]
# Rule: attn[SECOND, FIRST]

prompt = "A screen reader is"
tokens = model.to_str_tokens(prompt)
print(list(enumerate(tokens)))  # verify indices
logits, cache = model.run_with_cache(prompt)

threshold = 0.1
rows = []

for layer in range(model.cfg.n_layers):
    attention = cache["pattern", layer]
    for head in range(model.cfg.n_heads):
        attn = attention[0, head]
        reader_idx = 3
        screen_idx = 2
        score = attn[reader_idx, screen_idx].item()
        if score > threshold:
            rows.append({
                "layer": layer,
                "head": head,
                "binding_score": round(score, 4)
            })

with open("../../results/pythia/extended/2.8b_screenReader_attention_binding.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["layer", "head", "binding_score"])
    writer.writeheader()
    writer.writerows(rows)

print(f"Found {len(rows)} heads above threshold {threshold}")
print("\nTop 10 by binding strength:")
sorted_rows = sorted(rows, key=lambda x: x["binding_score"], reverse=True)
for row in sorted_rows[:10]:
    print(f"Layer {row['layer']:2d}, Head {row['head']:2d}: {row['binding_score']}")
print("Saved to 2.8b_screenReader_attention_binding.csv")

[(0, '<|endoftext|>'), (1, 'A'), (2, ' screen'), (3, ' reader'), (4, ' is')]
Found 101 heads above threshold 0.1

Top 10 by binding strength:
Layer  1, Head 12: 0.9909
Layer  1, Head 29: 0.984
Layer  1, Head  6: 0.9815
Layer  0, Head 21: 0.9726
Layer  1, Head 17: 0.9719
Layer  1, Head 25: 0.9483
Layer  1, Head 11: 0.9268
Layer  1, Head 22: 0.9058
Layer 29, Head  7: 0.9019
Layer  4, Head 16: 0.8896
Saved to 2.8b_screenReader_attention_binding.csv


##### Circuitsvis
**Layer 1, Head 12**

In [32]:
# Layer 1, Head 12
layer = 1
attention = cache["pattern", layer]
cv.attention.attention_patterns(tokens=tokens, attention=attention[0])

#### Alt Text (text to alt)

In [33]:
# DIRECTION: "text" looks at "alt" = attn[text_idx, alt_idx]
# Rule: attn[SECOND, FIRST]

prompt = "An image needs alt text to be accessible"
tokens = model.to_str_tokens(prompt)
print(list(enumerate(tokens)))  # verify indices
logits, cache = model.run_with_cache(prompt)

threshold = 0.1
rows = []

for layer in range(model.cfg.n_layers):
    attention = cache["pattern", layer]
    for head in range(model.cfg.n_heads):
        attn = attention[0, head]
        text_idx = 5
        alt_idx = 4
        score = attn[text_idx, alt_idx].item()
        if score > threshold:
            rows.append({
                "layer": layer,
                "head": head,
                "binding_score": round(score, 4)
            })

with open("../../results/pythia/extended/2.8b_altText_attention_binding.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["layer", "head", "binding_score"])
    writer.writeheader()
    writer.writerows(rows)

print(f"Found {len(rows)} heads above threshold {threshold}")
print("\nTop 10 by binding strength:")
sorted_rows = sorted(rows, key=lambda x: x["binding_score"], reverse=True)
for row in sorted_rows[:10]:
    print(f"Layer {row['layer']:2d}, Head {row['head']:2d}: {row['binding_score']}")
print("Saved to 2.8b_altText_attention_binding.csv")

[(0, '<|endoftext|>'), (1, 'An'), (2, ' image'), (3, ' needs'), (4, ' alt'), (5, ' text'), (6, ' to'), (7, ' be'), (8, ' accessible')]
Found 200 heads above threshold 0.1

Top 10 by binding strength:
Layer  1, Head 25: 0.9899
Layer  3, Head  1: 0.9875
Layer  1, Head 12: 0.9844
Layer  1, Head 16: 0.9598
Layer  3, Head 24: 0.945
Layer  3, Head 20: 0.9275
Layer 21, Head  6: 0.9165
Layer  1, Head  6: 0.8808
Layer  4, Head 16: 0.8742
Layer 25, Head  2: 0.8689
Saved to 2.8b_altText_attention_binding.csv


##### Circuitsvis
**Layer 1, Head 25**

In [34]:
# Layer 1, Head 25
layer = 1
attention = cache["pattern", layer]
cv.attention.attention_patterns(tokens=tokens, attention=attention[0])

#### Skip Link (link to skip)

In [35]:
# DIRECTION: "link" looks at "skip" = attn[link_idx, skip_idx]
# Rule: attn[SECOND, FIRST]

prompt = "Use a skip link to bypass navigation"
tokens = model.to_str_tokens(prompt)
print(list(enumerate(tokens)))  # verify indices
logits, cache = model.run_with_cache(prompt)

threshold = 0.1
rows = []

for layer in range(model.cfg.n_layers):
    attention = cache["pattern", layer]
    for head in range(model.cfg.n_heads):
        attn = attention[0, head]
        link_idx = 4
        skip_idx = 3
        score = attn[link_idx, skip_idx].item()
        if score > threshold:
            rows.append({
                "layer": layer,
                "head": head,
                "binding_score": round(score, 4)
            })

with open("../../results/pythia/extended/2.8b_skipLink_attention_binding.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["layer", "head", "binding_score"])
    writer.writeheader()
    writer.writerows(rows)

print(f"Found {len(rows)} heads above threshold {threshold}")
print("\nTop 10 by binding strength:")
sorted_rows = sorted(rows, key=lambda x: x["binding_score"], reverse=True)
for row in sorted_rows[:10]:
    print(f"Layer {row['layer']:2d}, Head {row['head']:2d}: {row['binding_score']}")
print("Saved to 2.8b_skipLink_attention_binding.csv")

[(0, '<|endoftext|>'), (1, 'Use'), (2, ' a'), (3, ' skip'), (4, ' link'), (5, ' to'), (6, ' bypass'), (7, ' navigation')]
Found 182 heads above threshold 0.1

Top 10 by binding strength:
Layer 30, Head  4: 0.9949
Layer  1, Head 12: 0.9839
Layer 30, Head 29: 0.9671
Layer 28, Head  1: 0.9452
Layer  1, Head  6: 0.9212
Layer  4, Head 16: 0.8858
Layer  3, Head 23: 0.8716
Layer 27, Head  9: 0.8458
Layer  6, Head  6: 0.8367
Layer 27, Head 28: 0.8273
Saved to 2.8b_skipLink_attention_binding.csv


##### Circuitsvis
**Layer 30, Head 4**

In [36]:
# Layer 30, Head 4
layer = 30
attention = cache["pattern", layer]
cv.attention.attention_patterns(tokens=tokens, attention=attention[0])

### Delete Model & Clear Cache

In [37]:
# Run this between models
del model
del cache
gc.collect()
torch.cuda.empty_cache()
print("Memory cleared")

Memory cleared
